In [ ]:
# 필요 라이브러리 설치
!pip install langchain chromadb openai tiktoken

In [ ]:
# 필요한 라이브러리 임포트
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.text_splitter import CharacterTextSplitter
from langchain.llms import OpenAI
from langchain.chains import RetrievalQA
import tiktoken
import time

In [ ]:
# OpenAI API 키
import os
os.environ["OPENAI_API_KEY"] = "FINAL_TEAM3"

In [ ]:
# 토큰 수 계산 함수
def num_tokens_from_string(string: str, encoding_name: str) -> int:
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

# 임베딩 생성 시간 및 토큰 사용량 예상 함수
def estimate_embedding_cost(texts, model="text-embedding-ada-002"):
    total_tokens = sum(num_tokens_from_string(text, "cl100k_base") for text in texts)
    estimated_time = total_tokens * 0.0001  # 대략적인 추정값, 실제로는 다를 수 있음
    estimated_cost = total_tokens * 0.0001 / 1000  # $0.0001 per 1K tokens

    return total_tokens, estimated_time, estimated_cost

In [ ]:
# 샘플 문서 생성
documents = [
    "The sky is blue.",
    "Grass is green.",
    "The sun is yellow.",
    "Water is clear and refreshing.",
    "Trees have leaves and branches."
]

# 문서 분할
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.create_documents(documents)
text_contents = [doc.page_content for doc in texts]

In [ ]:
# 임베딩 생성 전 예상 비용 계산
total_tokens, estimated_time, estimated_cost = estimate_embedding_cost(text_contents)

print(f"예상 토큰 수: {total_tokens}")
print(f"예상 소요 시간: {estimated_time:.2f}초")
print(f"예상 비용: ${estimated_cost:.6f}")

# 사용자 확인
user_input = input("계속 진행하시겠습니까? (y/n): ")

if user_input.lower() != 'y':
    print("프로그램을 종료합니다.")
    exit()

# 임베딩 생성 및 Chroma DB에 저장
print("임베딩 생성을 시작합니다...")
start_time = time.time()
embeddings = OpenAIEmbeddings()
db = Chroma.from_documents(texts, embeddings)
end_time = time.time()

print(f"실제 소요 시간: {end_time - start_time:.2f}초")

In [ ]:
# 검색 기능 생성
retriever = db.as_retriever()

# LLM 설정
llm = OpenAI()

# RetrievalQA 체인 생성
qa = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever)

# 질문 답변
query = "What color is the sky?"
result = qa.run(query)
print(result)

In [ ]:
# 다른 질문 시도
query = "Tell me about trees."
result = qa.run(query)
print(result)

# Chroma DB 지속성 (선택사항)
persist_directory = "chroma_db"
db = Chroma.from_documents(texts, embeddings, persist_directory=persist_directory)
db.persist()

In [ ]:
# 나중에 저장된 DB 로드
loaded_db = Chroma(persist_directory=persist_directory, embedding_function=embeddings)